# Segmentation Workflow Walkthrough

This notebook walks through the dependency-light segmentation scaffold before any SegFormer fine-tuning. It prepares dataset templates, checks mask metrics, records environment details, and keeps real training behind an explicit flag.

Use the notebook to understand the reporting contract first: image-mask CSV shape, IoU and Dice metrics, dry-run training command, saved metrics files, and dataset provenance. Real medical-image experiments should be run from the scripts after dataset terms, citation requirements, and output privacy have been reviewed.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


CHAPTER_DIR = find_chapter_dir("segmentation_medical_workflow.py", "chapter_segmentation_cnn_transformers")
WORKFLOW_SCRIPT = CHAPTER_DIR / "segmentation_medical_workflow.py"
TRAIN_SCRIPT = CHAPTER_DIR / "segformer_medical_finetune.py"


def run_script(script: Path, *args: object) -> subprocess.CompletedProcess[str]:
    cmd = [sys.executable, str(script), *map(str, args)]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=CHAPTER_DIR, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()
    return result


print(f"Chapter directory: {CHAPTER_DIR}")

## Dependency Checks

The workflow script is lightweight and should run in a minimal Python environment. The SegFormer training script separately reports whether optional ML packages are installed, so readers can distinguish a documentation or data-prep check from a real fine-tuning environment.

In [ ]:
run_script(WORKFLOW_SCRIPT, "check-deps", "--allow-missing-deps")
run_script(TRAIN_SCRIPT, "dependency-check", "--allow-missing-deps")


## Run The Smoke Workflow

The smoke command writes tiny text masks, computes IoU and Dice, emits a dry-run training command, and summarizes a fake metrics file. This is a pipeline check, not medical evidence; use it to learn the artifact names and metric fields before touching real images.

In [ ]:
smoke_dir = CHAPTER_DIR / "artifacts" / "segmentation_smoke"
run_script(WORKFLOW_SCRIPT, "smoke", "--output-dir", smoke_dir)


## Inspect Smoke Artifacts

The listed files show the reporting contract expected from real segmentation runs. Check the metric JSON, generated command, and directory layout so later experiment outputs can be compared without guessing which files came from which run.

In [ ]:
for path in sorted(smoke_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(smoke_dir))

metrics_path = smoke_dir / "metrics" / "metric_example.json"
if metrics_path.exists():
    print("
Metric example:")
    print(metrics_path.read_text(encoding="utf-8"))


## Dataset Template And Dry-Run Command

Use this cell when preparing a real image-mask CSV with paths, splits, and dataset provenance. The generated training command should be reviewed before launching fine-tuning, especially the model checkpoint, image size, output directory, and whether a test split is being held back.

In [ ]:
template_dir = CHAPTER_DIR / "artifacts" / "segmentation_template"
command_dir = CHAPTER_DIR / "artifacts" / "segmentation_train_command"
run_script(WORKFLOW_SCRIPT, "make-template", "--output-dir", template_dir)
run_script(
    WORKFLOW_SCRIPT,
    "train-command",
    "--data-root", template_dir,
    "--model", "nvidia/segformer-b0-finetuned-ade-512-512",
    "--image-size", "256",
    "--epochs", "5",
    "--output-dir", command_dir,
    "--dry-run",
)


## Optional Real Training

Real training needs a prepared `pairs.csv`, dataset provenance, optional segmentation dependencies, and enough compute for the selected image size and batch size. Keep test split evaluation until after model and threshold selection, and avoid publishing qualitative grids or derived masks unless the upstream dataset terms allow it.

In [ ]:
RUN_TRAINING = False
pairs_csv = Path("/path/to/kvasir-seg/pairs.csv")
data_root = Path("/path/to/kvasir-seg")

if RUN_TRAINING:
    run_script(
        TRAIN_SCRIPT,
        "train",
        "--pairs-csv", pairs_csv,
        "--data-root", data_root,
        "--output-dir", CHAPTER_DIR / "runs" / "kvasir-segformer-b0-256",
        "--model-name", "nvidia/segformer-b0-finetuned-ade-512-512",
        "--image-size", "256",
        "--epochs", "5",
        "--batch-size", "4",
        "--learning-rate", "5e-5",
        "--thresholds", "0.3", "0.5", "0.7",
        "--amp",
    )
else:
    print("Set RUN_TRAINING = True only after preparing a licensed dataset and split CSV.")


## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.